<pre>
- Monóxido de carbono   -> Unidade de medida de retorno kg/kg
</pre>

Unidade nativa: kg/kg-¹ (razão de mistura em massa / mass mixing ratio)
O que significa: Quilogramas do gás poluidor por quilograma de ar.

Nota de conversão: Na prática e em estudos de qualidade do ar, costuma-se converter kg/kg para fração em volume em partes por milhão (ppm) ou partes por bilhão (ppb) utilizando a massa molar do gás e do ar seco (≈28,96 g/mol) que é igual a 

In [ ]:
import cdsapi
import os, sys
import xarray as xr
from pyspark.sql import functions as F

In [ ]:
# Cria uma conexão SPARK

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


In [ ]:
def get_EAC4_(year):
    dataset = "cams-global-reanalysis-eac4"
    request = {
        "variable": [
            "carbon_monoxide"
        ],
        "pressure_level": ["1000"],
        "date": [f"{year}-01-01/{year}-12-31"],
        "time": ["06:00"],
        "data_format": "netcdf",
        "area": [6, -74, -34, -38]
    }

    client = cdsapi.Client(
        url = "https://ads.atmosphere.copernicus.eu/api",
        key = "34161618-bf6b-41ca-9272-50b917f789b9"
    )

    ret_download = client.retrieve(dataset, request).download()
    return ret_download

def convert_nc_to_spark_dataframe(path, file_name):
    with xr.open_dataset(f"{path}\\{file_name}"
                        ,engine="netcdf4"
                        ,chunks={"time": 365
                                ,"latitude": 100
                                ,"longitude": 100 }
                        ) as ds:
        # Transforma o Dataset em um Spark Dataframe
        df_dask             = ds.to_dask_dataframe()
        df_dask_c           = df_dask.compute()
        df_monoxido_carbono = spark.createDataFrame(df_dask_c)

    return df_monoxido_carbono

def convert_unit(df_monoxido_carbono):
    # Converte o valor do monoxido de carbodo de kg/kg-¹ para ppb (partes por bilhão)
    M_AR = 28.9644 # g/mol
    M_CO = 28.0101 # g/mol
    FATOR_CONVERSAO = (M_AR / M_CO) * 1e9  # ~ 1.03407e9

    drop_cols = ["valid_time", "pressure_level", "co"]

    df_monoxido_carbono_ppb = \
        (df_monoxido_carbono
            .withColumns({"data_medicao": F.col("valid_time").cast("date")
                        ,"indicador": F.lit("Poluição do ar - CO (ppb)") 
                        ,"valor": (F.col("co") * F.lit(FATOR_CONVERSAO)).cast("double")
                        ,"unidade_medida": F.lit("ppb")})
            .drop(*drop_cols)
    )
    return df_monoxido_carbono_ppb

def write_data_csv(df_monoxido_carbono_ppb, write_path, file_name):
    
    df_monoxido_carbono_ppb.toPandas().to_csv(f"{write_path}\{file_name}")


In [ ]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)
print(DATA_PATH_ROOT)

# "C:\Marco Conti\Projetos\Dados\EAC4-poluicao\csv\EAC4_co_2005.nc"

In [ ]:
years_process = [2005, 2006, 2007, 2008, 2009
                ,2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019
                ,2020, 2021, 2022, 2023, 2024, 2026]

for year in years_process:

    # recupera os dados de co2 
    # ret_download = get_EAC4_co(year)

    df_monoxido_carbono = \
        convert_nc_to_spark_dataframe(f"{DATA_PATH_ROOT}\EAC4-poluicao", f"EAC4_co_{year}.nc")

    df_monoxido_carbono_ppb = convert_unit(df_monoxido_carbono)

    csv_path = r"{DATA_PATH_ROOT}\EAC4-poluicao\arquivos"
    csv_file_name

    write_data_csv(df_monoxido_carbono_ppb, csv_path, csv_file_name)

    # os.rename(ret_download, f"{DATA_PATH_ROOT}\EAC4-poluicao\arquivos_csv\EAC4_co_{year}.nc")

    print(f"Download completed: {ret_download}", "\n")

In [ ]:
# df_monoxido_carbono_ppb.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_monoxido_carbono.csv", index=False)

df_monoxido_carbono_ppb.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_monoxido_carbono.parquet")


In [ ]:
df_monoxido_carbono_parquet = \
    spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_monoxido_carbono.parquet")

df_monoxido_carbono_parquet.printSchema()
df_monoxido_carbono_parquet.show(10, False)

In [ ]:
os.remove(r"C:\Marco Conti\Projetos\MAIS-v2\Poluicao\{file_name}".format(file_name = ret_download))